# Question-2: CycleGAN for Person Face Sketches
Unpaired photo ↔ sketch translation (Zhu et al., 2017), trained end-to-end on
[Person Face Sketches](https://www.kaggle.com/datasets/almightyj/person-face-sketches).

**Runtime → Change runtime type → GPU.** Checkpoints go to Google Drive. After a disconnect, run the cells again from the top and training picks up from the last saved epoch.

**Submitting:** the graded record is this notebook *with its outputs*. In Colab, open Edit → Notebook settings and make sure "Omit code cell output when saving" is **off**. When everything has run, use File → Download → .ipynb and replace `colab_run.ipynb` in the repo with it. Training logs are also written to Drive, and the *Full log* cells below print them into the notebook, so a run split across several sessions still appears in full.

**Run all:** Runtime → Run all runs the whole notebook start to finish, in about 2–3.5 hours on a T4. The only interaction is the Google Drive permission popup at the start, and allowing the file download at the end. Keep the tab open while it runs. If a step fails, the cell right after it stops with a message saying which step.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again"

## 2. Drive + code
Upload this `q2/` folder to `MyDrive/as2/q2` first. If you use a different location, change `CODE_DIR`.

Each script's console output is also appended to `cyclegan_ckpt/logs/<step>.log` on Drive, so it survives a disconnect.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CODE_DIR = '/content/drive/MyDrive/as2/q2'
CKPT = '/content/drive/MyDrive/cyclegan_ckpt'
%cd $CODE_DIR
!pip -q install -r requirements.txt
!mkdir -p "$CKPT/logs"

## 3. Dataset
`kagglehub` asks for your Kaggle credentials if it needs them. The second command prints how many photos and sketches the loader found. If it picked the wrong folders, set `PHOTO_DIR`/`SKETCH_DIR` below.

In [ ]:
import os, kagglehub
# Kaggle login without a prompt: add KAGGLE_USERNAME and KAGGLE_KEY in Colab's Secrets panel (key icon, left sidebar)
# and turn on notebook access. Public datasets usually download without them.
from google.colab import userdata
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("No Kaggle secrets set; trying an anonymous download")
DATA = kagglehub.dataset_download("almightyj/person-face-sketches")
print(DATA)
!find "$DATA" -maxdepth 3 -type d | head -20

In [ ]:
import os
# Train on train/ when the dataset has splits; test images must never be trained on.
ROOT = next((f"{DATA}/{s}" for s in ("train", "Train") if os.path.isdir(f"{DATA}/{s}")), DATA)
TEST_ROOT = next((f"{DATA}/{s}" for s in ("test", "Test", "val", "Val") if os.path.isdir(f"{DATA}/{s}")), None)
print("train root:", ROOT, "| test root:", TEST_ROOT)
# Leave empty to auto-split by folder name, or set explicit training folders, e.g. f"{ROOT}/photos"
PHOTO_DIR, SKETCH_DIR = "", ""
DATA_FLAGS = f'--data_dir "{ROOT}"' + (f' --photo_dir "{PHOTO_DIR}" --sketch_dir "{SKETCH_DIR}"' if PHOTO_DIR and SKETCH_DIR else "")
!python -u dataset.py $DATA_FLAGS 2>&1 | tee -a "$CKPT/logs/dataset.log"
# Stop here (not hours later) if the photo/sketch split looks wrong
from dataset import resolve_domains
train_photos, train_sketches = resolve_domains(ROOT, PHOTO_DIR or None, SKETCH_DIR or None)
assert len(train_photos) > 100 and len(train_sketches) > 100, "too few images found: set PHOTO_DIR/SKETCH_DIR above"

## 4. Hyperparameter tuning (short runs)
Each setting below trains for 3 epochs at 128 px on 1,000 images per domain, with its own checkpoint dir. We compare
the cycle-consistency loss (lower means better reconstruction) and look at the sample grids. GAN losses alone don't say much about quality.

Note that batch size 4 makes 4× fewer updates in 3 epochs than batch size 1, so its loss is a little higher for that reason alone. Judge it partly on the sample grids.

In [ ]:
TUNE = {
    "lr2e-4_bs1": "--lr 2e-4 --batch_size 1",
    "lr1e-4_bs1": "--lr 1e-4 --batch_size 1",
    "lr2e-4_bs4": "--lr 2e-4 --batch_size 4",
}
for name, flags in TUNE.items():
    !python -u train.py $DATA_FLAGS --checkpoint_dir "$CKPT/tune_$name" {flags} \
        --epochs 3 --image_size 128 --n_blocks 6 --max_samples 1000 --snapshot_every 0 --amp --log_every 200 2>&1 | tee -a "$CKPT/logs/tune_{name}.log"
import os
for name in TUNE:
    assert os.path.exists(f"{CKPT}/tune_{name}/history.csv"), f"tuning run {name} failed: see the output above"

### Full tuning logs (all sessions)

In [ ]:
for name in TUNE:
    print(f"==================== {name} ====================")
    !cat "$CKPT/logs/tune_{name}.log"

In [ ]:
import pandas as pd
rows = []
for name in TUNE:
    h = pd.read_csv(f"{CKPT}/tune_{name}/history.csv")
    rows.append({"run": name, **h.iloc[-1][["loss_G", "loss_D", "loss_cycle", "loss_idt", "seconds"]].to_dict()})
tuning = pd.DataFrame(rows).sort_values("loss_cycle")
display(tuning)
BEST_RUN = tuning.iloc[0]["run"]
print("picked for full training:", BEST_RUN, "->", TUNE[BEST_RUN])

In [ ]:
import glob
from IPython.display import Image, display
for name in TUNE:
    print(name); display(Image(sorted(glob.glob(f"{CKPT}/tune_{name}/samples/epoch_*.png"))[-1]))

## 5. Full training
Uses the lr/batch size picked by the tuning step (`BEST_RUN`). To force a setting, replace the first line with e.g. `BEST = "--lr 2e-4 --batch_size 1"`.

**Session time:** the defaults below (256 px, 1,500 images per domain, 20 epochs) are sized so the whole notebook fits in one Colab session, roughly 2–3 hours on a T4. For better quality, raise `EPOCHS`/`MAX_SAMPLES` if your Colab plan allows longer sessions.

**Disconnects:** if Colab disconnects anyway, just **Run all** again. Finished tuning runs are skipped and training resumes from `latest.pth`.

**Drive space:** `latest.pth` is about 340 MB and every 5th epoch adds a 91 MB snapshot. Empty Drive's Trash now and then.

In [ ]:
BEST = TUNE[BEST_RUN]
EPOCHS, MAX_SAMPLES, IMAGE_SIZE = 20, 1500, 256
!python -u train.py $DATA_FLAGS --checkpoint_dir "$CKPT/main" {BEST} --epochs {EPOCHS} --max_samples {MAX_SAMPLES} --image_size {IMAGE_SIZE} --snapshot_every 5 --amp --num_workers 2 2>&1 | tee -a "$CKPT/logs/train_main.log"
import os
assert os.path.exists(f"{CKPT}/main/latest.pth"), "training failed: see the output above"

### Full training log (all sessions)
Training usually spans several Colab sessions, and the cell above only keeps the last one. This prints every session's progress, with timestamps, from the log on Drive.

In [ ]:
!cat "$CKPT/logs/train_main.log"

## 6. Monitor training

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display
h = pd.read_csv(f"{CKPT}/main/history.csv")
h.plot(x="epoch", y=["loss_G", "loss_D", "loss_cycle", "loss_idt"], figsize=(9, 4), title="CycleGAN losses")
display(h.tail())
latest = sorted(glob.glob(f"{CKPT}/main/samples/epoch_*.png"))[-1]
print(latest, "(rows: photo, ->sketch, sketch, ->photo)"); display(Image(latest))

## 7. Test both directions
Uses the auto-detector: a photo becomes a sketch and a sketch becomes a photo.

In [ ]:
from IPython.display import Image, display
from dataset import resolve_domains
try:
    test_photos, test_sketches = resolve_domains(TEST_ROOT)
except (TypeError, FileNotFoundError, ValueError) as err:  # no test split, or its folders aren't auto-detectable
    print(f"WARNING: using training images for this test ({err}); results may be optimistic")
    test_photos, test_sketches = train_photos, train_sketches
for i, src in enumerate([test_photos[0], test_sketches[0]]):
    !python -u inference.py --checkpoint "$CKPT/main/latest.pth" --input "{src}" --output /content/out_{i}.png 2>&1 | tee -a "$CKPT/logs/inference.log"
    display(Image(str(src), width=256), Image(f"/content/out_{i}.png", width=256))

### Sketch/photo auto-detection accuracy
The UI picks the direction with `inference.is_sketch`. This measures how often it is right on held-out images. Report the number.

In [ ]:
import random
from PIL import Image as PILImage
from inference import is_sketch
rng = random.Random(0)
P, S = rng.sample(test_photos, min(500, len(test_photos))), rng.sample(test_sketches, min(500, len(test_sketches)))
photo_ok = sum(not is_sketch(PILImage.open(p)) for p in P)
sketch_ok = sum(is_sketch(PILImage.open(p)) for p in S)
print(f"photos correctly detected:   {photo_ok}/{len(P)}")
print(f"sketches correctly detected: {sketch_ok}/{len(S)}")
print(f"overall accuracy: {(photo_ok + sketch_ok) / (len(P) + len(S)):.1%}")

## 8. Export the model for the local UI
Saves only the two generators from the latest epoch (about 91 MB) as `cyclegan_final.pth`, then downloads it. Put the file in `q2/checkpoints/` on your computer and run `./run_local.sh`. See *Run the UI locally* in the README.

In [ ]:
import torch
state = torch.load(f"{CKPT}/main/latest.pth", map_location="cpu", weights_only=False)
torch.save({k: state[k] for k in ("epoch", "args", "G_AB", "G_BA")}, f"{CKPT}/cyclegan_final.pth")
print(f"exported epoch {state['epoch']} -> {CKPT}/cyclegan_final.pth")
from google.colab import files; files.download(f"{CKPT}/cyclegan_final.pth")  # or grab it from Drive

## 9. Before closing the session
Flush Drive so the last checkpoint is fully written.

In [ ]:
from google.colab import drive; drive.flush_and_unmount()